# 03. Scale Ablation Study

**Project:** KPI-RAG: Explainable Root-Cause Analysis for 5G Networks
**Stage:** Phase 1 — ScaleAblation Module

This notebook answers RQ1 and RQ3 from the proposal by comparing three feature configurations
on both the binary anomaly detector and the 11-class fault classifier.

**Research Questions:**
- **RQ1:** Does scale-preserving feature engineering improve anomaly detection F1 and fault classification F1-macro vs normalized-feature and statistics-only baselines?
- **RQ3:** Which feature configuration offers the best performance-to-cost tradeoff?

**Three Configurations:**

| Config | Dims | Components | Description |
|---|---|---|---|
| A | 64 | Statistics only | mean, std, min, max per 16 channels |
| B | 320 | Statistics + Patchwise Scale | Config A + 8 patches x mean+std x 16 channels |
| C | 582 | Full vector | Config B + first-order differences + categorical encoding |

**Critical design rule:** Patchwise scale features are never normalized.
Absolute KPI magnitude is the diagnostic signal — normalization destroys it.

**Output:** Ablation results table, bar chart, and three feature matrices saved to data folder.

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

import sys
sys.path.append(r'C:\Users\DELL\Desktop\kpi_rag\src')

COLORS = {
    'normal':     '#2E6DB4',
    'anomaly':    '#C0392B',
    'jamming':    '#E67E22',
    'synthetic':  '#85929E',
    'axis':       '#2C3E50',
    'grid':       '#F2F4F6',
    'background': '#FFFFFF',
}

DATA_DIR = r'C:\Users\DELL\Desktop\kpi_rag\data'

# Feature vector index boundaries
STATS_END   = 64
SCALE_END   = 320
DIFFS_END   = 576
CATS_END    = 582

# Random Forest configuration — same across all configs for fair comparison
RF_PARAMS = {
    'n_estimators':  300,
    'max_depth':     None,
    'min_samples_leaf': 2,
    'n_jobs':        -1,
    'random_state':  42,
    'class_weight':  'balanced',
}

print('Configuration loaded.')
print(f'Config A: dims [0:{STATS_END}]')
print(f'Config B: dims [0:{SCALE_END}]')
print(f'Config C: dims [0:{CATS_END}]')

## 2. Load Data

Loads the full 582-dim feature matrix and labels from Notebook 02.
The train/test split indices from Notebook 02b are applied directly.

In [ ]:
print('Loading feature matrix and labels...')

X_full       = np.load(os.path.join(DATA_DIR, 'X_features.npy'))
y_binary     = np.load(os.path.join(DATA_DIR, 'y_binary.npy'))
y_multiclass = np.load(os.path.join(DATA_DIR, 'y_multiclass.npy'))
train_idx    = np.load(os.path.join(DATA_DIR, 'train_idx.npy'))
test_idx     = np.load(os.path.join(DATA_DIR, 'test_idx.npy'))

with open(os.path.join(DATA_DIR, 'type_to_int.json')) as f:
    type_to_int = json.load(f)
int_to_type = {v: k for k, v in type_to_int.items()}

print(f'X_full shape:       {X_full.shape}')
print(f'y_binary shape:     {y_binary.shape}')
print(f'y_multiclass shape: {y_multiclass.shape}')
print(f'Train samples:      {len(train_idx):,}')
print(f'Test samples:       {len(test_idx):,}')
print(f'Fault types:        {len(type_to_int) - 1}')

## 3. Build Ablated Feature Matrices

Slices the full 582-dim matrix into three configurations.
No recomputation is needed — each config is a column slice of the full matrix.

In [ ]:
# Config A — Statistics only (64 dims)
X_A = X_full[:, :STATS_END]

# Config B — Statistics + Patchwise Scale (320 dims)
X_B = X_full[:, :SCALE_END]

# Config C — Full vector (582 dims)
X_C = X_full[:, :CATS_END]

print('Feature matrix slices:')
print(f'  Config A shape: {X_A.shape}')
print(f'  Config B shape: {X_B.shape}')
print(f'  Config C shape: {X_C.shape}')

# Apply train/test split to each config
X_A_train, X_A_test = X_A[train_idx], X_A[test_idx]
X_B_train, X_B_test = X_B[train_idx], X_B[test_idx]
X_C_train, X_C_test = X_C[train_idx], X_C[test_idx]

y_bin_train = y_binary[train_idx]
y_bin_test  = y_binary[test_idx]
y_mc_train  = y_multiclass[train_idx]
y_mc_test   = y_multiclass[test_idx]

print()
print(f'Train binary:     Normal={np.bincount(y_bin_train)[0]:,}  Anomaly={np.bincount(y_bin_train)[1]:,}')
print(f'Test binary:      Normal={np.bincount(y_bin_test)[0]:,}   Anomaly={np.bincount(y_bin_test)[1]:,}')

## 4. Save Ablated Feature Matrices

Saves the three feature matrices to the data folder for sharing with P2 (Detector module).

In [ ]:
np.save(os.path.join(DATA_DIR, 'X_config_A.npy'), X_A)
np.save(os.path.join(DATA_DIR, 'X_config_B.npy'), X_B)
np.save(os.path.join(DATA_DIR, 'X_config_C.npy'), X_C)

print('Saved ablated feature matrices:')
print(f'  X_config_A.npy   {X_A.shape}  — 64-dim  statistics only')
print(f'  X_config_B.npy   {X_B.shape}  — 320-dim statistics + scale')
print(f'  X_config_C.npy   {X_C.shape}  — 582-dim full vector')

## 5. Training and Evaluation Functions

Defines reusable functions for training and evaluating Random Forest models.
The same RF hyperparameters are used across all configs to ensure a fair comparison.

In [ ]:
def train_and_evaluate_binary(X_train, X_test, y_train, y_test, config_name):
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    f1        = f1_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall    = recall_score(y_test, y_pred, pos_label=1, zero_division=0)

    print(f'  Config {config_name} — Binary Detector:')
    print(f'    F1:        {f1:.4f}')
    print(f'    Precision: {precision:.4f}')
    print(f'    Recall:    {recall:.4f}')

    return {'config': config_name, 'f1': f1, 'precision': precision,
            'recall': recall, 'model': rf}


def train_and_evaluate_multiclass(X_train, X_test, y_train, y_test,
                                   config_name, int_to_type):
    # Use only anomalous samples — exclude Normal (class 0) and Jamming
    JAMMING_CLASS = type_to_int.get('Jamming', -1)

    anomaly_mask_train = (y_train > 0) & (y_train != JAMMING_CLASS)
    anomaly_mask_test  = (y_test  > 0) & (y_test  != JAMMING_CLASS)

    X_tr = X_train[anomaly_mask_train]
    y_tr = y_train[anomaly_mask_train]
    X_te = X_test[anomaly_mask_test]
    y_te = y_test[anomaly_mask_test]

    if len(np.unique(y_tr)) < 2:
        print(f'  Config {config_name} — Fault Classifier: insufficient classes in train set.')
        return None

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)

    f1_macro = f1_score(y_te, y_pred, average='macro', zero_division=0)
    accuracy = (y_pred == y_te).mean()

    print(f'  Config {config_name} — Fault Classifier:')
    print(f'    F1-macro:  {f1_macro:.4f}')
    print(f'    Accuracy:  {accuracy:.4f}')
    print(f'    Train samples (excl. Jamming): {len(X_tr):,}')
    print(f'    Test samples  (excl. Jamming): {len(X_te):,}')

    return {'config': config_name, 'f1_macro': f1_macro,
            'accuracy': accuracy, 'model': rf}

## 6. Run Ablation — Binary Anomaly Detector

Trains Model 1 (binary detector) on all three configs and records F1, precision, and recall.

In [ ]:
print('Running ablation — Binary Anomaly Detector')
print('=' * 50)
print('Note: this may take 5-10 minutes per config.')
print()

binary_results = []

for config_name, X_tr, X_te in [
    ('A (64-dim)',  X_A_train, X_A_test),
    ('B (320-dim)', X_B_train, X_B_test),
    ('C (582-dim)', X_C_train, X_C_test),
]:
    result = train_and_evaluate_binary(X_tr, X_te, y_bin_train, y_bin_test, config_name)
    binary_results.append(result)
    print()

print('Binary ablation complete.')

## 7. Run Ablation — Fault Classifier

Trains Model 2 (11-class fault classifier) on anomalous samples only.
Jamming is excluded from training — it is used as a held-out generalization probe.

In [ ]:
print('Running ablation — Fault Classifier (Jamming excluded from training)')
print('=' * 60)
print()

mc_results = []

for config_name, X_tr, X_te in [
    ('A (64-dim)',  X_A_train, X_A_test),
    ('B (320-dim)', X_B_train, X_B_test),
    ('C (582-dim)', X_C_train, X_C_test),
]:
    result = train_and_evaluate_multiclass(
        X_tr, X_te, y_mc_train, y_mc_test, config_name, int_to_type
    )
    if result:
        mc_results.append(result)
    print()

print('Fault classifier ablation complete.')

## 8. Jamming Generalization Probe

Evaluates the Config C fault classifier on Jamming samples it has never seen during training.
This tests whether the model learned generalizable fault representations.

In [ ]:
print('Jamming Generalization Probe')
print('=' * 40)

JAMMING_CLASS = type_to_int.get('Jamming', -1)

# Get the Config C fault classifier
rf_C = [r['model'] for r in mc_results if 'C' in r['config']][0]

# Jamming test samples from the test set
jamming_mask_test = y_mc_test == JAMMING_CLASS
X_jamming_test    = X_C_test[jamming_mask_test]
y_jamming_test    = y_mc_test[jamming_mask_test]

if len(X_jamming_test) > 0:
    y_jamming_pred = rf_C.predict(X_jamming_test)
    jamming_f1  = f1_score(y_jamming_test, y_jamming_pred,
                            average='macro', zero_division=0)
    jamming_acc = (y_jamming_pred == y_jamming_test).mean()

    print(f'Jamming test samples:   {len(X_jamming_test)}')
    print(f'Jamming F1 (macro):     {jamming_f1:.4f}')
    print(f'Jamming accuracy:       {jamming_acc:.4f}')
    print()
    print('Note: Jamming was completely excluded from training.')
    print('These results measure zero-exposure generalization.')
else:
    print('No Jamming samples found in test set.')

## 9. Results Summary Table

In [ ]:
print('Ablation Study Results')
print('=' * 70)
print(f'{"Config":<15} {"Dims":>6} {"Binary F1":>10} {"Precision":>10} {"Recall":>10}')
print('-' * 70)

for r in binary_results:
    print(f'{r["config"]:<15} {r["config"].split("(")[1].split("-")[0]:>6} '
          f'{r["f1"]:>10.4f} {r["precision"]:>10.4f} {r["recall"]:>10.4f}')

print()
print(f'{"Config":<15} {"Dims":>6} {"F1-macro":>10} {"Accuracy":>10}')
print('-' * 50)

for r in mc_results:
    print(f'{r["config"]:<15} {r["config"].split("(")[1].split("-")[0]:>6} '
          f'{r["f1_macro"]:>10.4f} {r["accuracy"]:>10.4f}')

## 10. Ablation Comparison Chart

In [ ]:
configs      = ['A\n(64-dim)', 'B\n(320-dim)', 'C\n(582-dim)']
binary_f1s   = [r['f1']       for r in binary_results]
mc_f1s       = [r['f1_macro'] for r in mc_results]

x     = np.arange(len(configs))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left — Binary Anomaly Detector
bars1 = axes[0].bar(x - width/2, binary_f1s, width,
                    color=COLORS['normal'], edgecolor='white', linewidth=0.8,
                    label='Binary F1')
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs, fontsize=10, color=COLORS['axis'])
axes[0].set_ylabel('F1 Score', fontsize=11, color=COLORS['axis'])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Binary Anomaly Detector — F1 by Config',
                  fontsize=12, color=COLORS['axis'], pad=10)
axes[0].set_facecolor(COLORS['grid'])
axes[0].grid(axis='y', color='white', linewidth=0.8)
axes[0].spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars1, binary_f1s):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=COLORS['axis'], fontweight='bold')

# Right — Fault Classifier
bars2 = axes[1].bar(x - width/2, mc_f1s, width,
                    color=COLORS['anomaly'], edgecolor='white', linewidth=0.8,
                    label='F1-macro')
axes[1].set_xticks(x)
axes[1].set_xticklabels(configs, fontsize=10, color=COLORS['axis'])
axes[1].set_ylabel('F1-macro Score', fontsize=11, color=COLORS['axis'])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Fault Classifier — F1-macro by Config',
                  fontsize=12, color=COLORS['axis'], pad=10)
axes[1].set_facecolor(COLORS['grid'])
axes[1].grid(axis='y', color='white', linewidth=0.8)
axes[1].spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars2, mc_f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, color=COLORS['axis'], fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ablation_comparison.png'), dpi=150)
plt.show()

print('Chart saved to data/ablation_comparison.png')

## 11. Validation Summary

In [ ]:
checks = {
    'Config A matrix saved':           os.path.exists(os.path.join(DATA_DIR, 'X_config_A.npy')),
    'Config B matrix saved':           os.path.exists(os.path.join(DATA_DIR, 'X_config_B.npy')),
    'Config C matrix saved':           os.path.exists(os.path.join(DATA_DIR, 'X_config_C.npy')),
    'Binary ablation has 3 results':   len(binary_results) == 3,
    'MC ablation has 3 results':       len(mc_results) == 3,
    'Config A dims correct (64)':      X_A.shape[1] == 64,
    'Config B dims correct (320)':     X_B.shape[1] == 320,
    'Config C dims correct (582)':     X_C.shape[1] == 582,
    'F1 improves A to C (binary)':     binary_results[2]['f1'] >= binary_results[0]['f1'],
    'F1 improves A to C (multiclass)': mc_results[2]['f1_macro'] >= mc_results[0]['f1_macro'],
}

all_passed = True
for check, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {check}')
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed. Scale ablation study complete.')
    print('Proceed to Phase 1 Documentation.')
else:
    print('One or more checks failed. Investigate before proceeding.')